In [1]:
import sounddevice as sd 
import numpy as np
import tqdm
import warnings
import joblib
import librosa
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from sklearn.pipeline import Pipeline
warnings.filterwarnings("ignore")

In [2]:
model = joblib.load('models/human_detector.pkl')

In [7]:
fs = 16000
duration = 10
print('Recording.....')
audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait()
print('Done!')
audio = audio.flatten()

Recording.....
Done!


In [4]:
SR          = 16000   # target sample rate
DURATION    = 3.0     # clip length in seconds (pad/trim)
N_MFCC      = 40      # number of MFCC coefficients
HOP_LENGTH  = 512
N_FFT       = 2048

In [5]:
def extract_features(y: np.ndarray, sr: int = fs) -> np.ndarray:
  """
  Extract a rich feature vector from one audio clip.
  Returns a 1-D numpy array of shape (feature_dim,).

  Features:
    MFCCs (40) × {mean, std, delta_mean, delta_std}    = 160
    Spectral centroid                 mean + std        =   2
    Spectral rolloff                  mean + std        =   2
    Spectral bandwidth                mean + std        =   2
    Spectral flatness                 mean + std        =   2
    Zero crossing rate                mean + std        =   2
    RMS energy                        mean + std        =   2
    Chroma (12)                       mean              =  12
    Fundamental frequency (F0)        mean + voiced_frac=   2
    HNR proxy (harmonics power ratio)                   =   1
    ─────────────────────────────────────────────────────────
    Total                                               = 187
  """
  feats = []

  # --- MFCCs + deltas ---
  mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                hop_length=HOP_LENGTH, n_fft=N_FFT)
  delta = librosa.feature.delta(mfcc)
  feats.extend(mfcc.mean(axis=1))
  feats.extend(mfcc.std(axis=1))
  feats.extend(delta.mean(axis=1))
  feats.extend(delta.std(axis=1))

  # --- Spectral features ---
  centroid  = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH)
  rolloff   = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH)
  bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=HOP_LENGTH)
  flatness  = librosa.feature.spectral_flatness(y=y, hop_length=HOP_LENGTH)

  for feat in [centroid, rolloff, bandwidth, flatness]:
      feats.append(feat.mean())
      feats.append(feat.std())

  # --- Zero Crossing Rate ---
  zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
  feats.append(zcr.mean())
  feats.append(zcr.std())

  # --- RMS energy ---
  rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
  feats.append(rms.mean())
  feats.append(rms.std())

  # --- Chroma ---
  chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=HOP_LENGTH)
  feats.extend(chroma.mean(axis=1))

  # --- Fundamental frequency (pyin) ---
  try:
      f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'),
                                          fmax=librosa.note_to_hz('C7'), sr=sr)
      f0_mean     = float(np.nanmean(f0)) if f0 is not None else 0.0
      voiced_frac = float(np.mean(voiced_flag)) if voiced_flag is not None else 0.0
  except Exception:
      f0_mean, voiced_frac = 0.0, 0.0
  feats.append(f0_mean)
  feats.append(voiced_frac)

  # --- HNR proxy: harmonic power / total power ---
  try:
      y_harmonic, _ = librosa.effects.hpss(y)
      hnr = float(np.sum(y_harmonic**2) / (np.sum(y**2) + 1e-8))
  except Exception:
      hnr = 0.0
  feats.append(hnr)

  return np.array(feats, dtype=np.float32)

In [8]:
features = extract_features(audio)
prediction = model.predict(features.reshape(1, -1))
print('Human' if prediction[0] == 1 else 'non_human')

Human


In [ ]:
import numpy as np
import sounddevice as sd
import tensorflow as tf
import tensorflow_hub as hub
import time

# Load YAMNet model
print("Loading YAMNet...")
model = hub.load("https://tfhub.dev/google/yamnet/1")

# Load class names
def load_class_names():
    class_map_path = tf.keras.utils.get_file(
        "yamnet_class_map.csv",
        "https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv"
    )
    with open(class_map_path) as f:
        lines = f.readlines()[1:]  # skip header
    return [line.strip().split(",", 2)[2].strip('"') for line in lines]

class_names = load_class_names()
print(f"Loaded {len(class_names)} classes\n")

# Audio settings
SAMPLE_RATE = 16000
FRAME_DURATION = 0.96   # seconds (YAMNet's native frame size)
FRAME_SAMPLES = int(SAMPLE_RATE * FRAME_DURATION)

def predict(audio_chunk):
    waveform = audio_chunk.flatten().astype(np.float32)
    
    start = time.perf_counter()
    scores, embeddings, spectrogram = model(waveform)
    elapsed = (time.perf_counter() - start) * 1000  # ms

    mean_scores = np.mean(scores.numpy(), axis=0)
    top_indices = np.argsort(mean_scores)[::-1][:3]

    print(f"[{elapsed:.1f}ms inference]")
    for i, idx in enumerate(top_indices):
        print(f"  {i+1}. {class_names[idx]:<35} {mean_scores[idx]:.3f}")
    print()

print(f"Listening on mic at {SAMPLE_RATE} Hz — press Ctrl+C to stop\n")

try:
    while True:
        # Record one frame of audio
        audio = sd.rec(
            frames=FRAME_SAMPLES,
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            blocking=True
        )
        predict(audio)

except KeyboardInterrupt:
    print("Stopped.")